# Climate Stance Detection — train all four detectors on Colab

Single-notebook launcher. Clones the project from GitHub, installs deps, runs `scripts/train_all.py`, then zips the results for download.

**Before running**
1. Edit `REPO_URL` in cell 2 to point at your GitHub repo (HTTPS clone URL).
2. Runtime → Change runtime type → **T4 GPU** (free tier).
3. Runtime → **Run all**.

Total wall time: ~35–50 min (BiLSTM ~2 min, lab transformer ~3 min, DistilBERT ~10 min, ClimateBERT ~12 min, plus deps + GloVe download).

At the end, `results.zip` shows up in the Files panel — right-click → Download. Drop it into the project folder on your Mac and unzip.

## 1. Verify the GPU

In [ ]:
!nvidia-smi

## 2. Clone the repo

Replace `REPO_URL` below with your GitHub HTTPS URL.

In [ ]:
REPO_URL = "https://github.com/<your-username>/nlp-climate-stance.git"

import os, subprocess
if not os.path.exists("/content/nlp-climate-stance"):
    subprocess.run(["git", "clone", REPO_URL, "/content/nlp-climate-stance"], check=True)
%cd /content/nlp-climate-stance
!git log --oneline -5

## 3. Install dependencies

Colab pre-installs most of these. We just top-up what's missing (transformers, datasets, evaluate, gensim, emoji).

In [ ]:
!pip install -q transformers datasets evaluate gensim emoji

## 4. Confirm GPU is visible to PyTorch and TensorFlow

In [ ]:
import torch, tensorflow as tf
print("torch CUDA:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
print("tensorflow GPUs:", tf.config.list_physical_devices("GPU"))

## 5. Train all four detectors

Runs `scripts/train_all.py` which executes:
1. `train_bilstm.py` — BiLSTM + GloVe (downloads ~128 MB GloVe on first run)
2. `train_lab_transformer.py` — Lab from-scratch transformer
3. `train_distilbert.py` — Fine-tuned DistilBERT (downloads ~250 MB checkpoint)
4. `train_climatebert.py` — Fine-tuned ClimateBERT (downloads ~310 MB checkpoint)

Each script appends a row to `results/metrics.csv` and saves predictions to `results/predictions/`.

If any single script fails, the run stops — copy the traceback so we can debug.

In [ ]:
!python scripts/train_all.py

## 6. Show the final metrics table

In [ ]:
import pandas as pd
pd.read_csv("results/metrics.csv")

## 7. Zip the results for download

We exclude `results/models/` — those are the trained HF model folders (~500 MB each) and we don't need them locally. Predictions, metrics, and figures are all small.

In [ ]:
!rm -f /content/results.zip
!cd /content/nlp-climate-stance && zip -r /content/results.zip \
    results/metrics.csv \
    results/predictions \
    results/figures
!ls -lh /content/results.zip

## 8. Download `results.zip`

Either:
- Files panel (folder icon on the left sidebar) → right-click `results.zip` → **Download**, or
- Run the cell below for an automatic browser download.

In [ ]:
from google.colab import files
files.download("/content/results.zip")